# Attribute Lookup, Properties, and Descriptors

This notebook has been rewritten to be slower, more reflective, and less template-like. Instead of treating **Attribute Lookup, Properties, and Descriptors** as a small isolated topic, the goal here is to treat it as part of Python's larger runtime story: objects are created, names are bound, frames execute bytecode, and the interpreter keeps following rules whether or not the source code looks simple from the outside.

If this topic ever felt a little too neat or too magical when explained quickly, that is exactly what this rewrite is trying to fix. We are going to keep asking two grounding questions: what changed in memory, and what instructions is the interpreter actually stepping through under the surface?

The point is not to drown in internals. The point is to make the language feel explainable. Once that happens, even advanced behavior starts feeling much less mysterious.


## Why this topic becomes easier once you watch the runtime carefully

When people struggle with **Attribute Lookup, Properties, and Descriptors**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


## Questions worth answering before we go any further

- Understand attribute lookup more deeply
- See properties as controlled attribute access
- Demystify descriptors
- Connect this topic to methods and framework internals


## What changes in memory while this code runs

An object often has its own instance dictionary, but the class may also define descriptors that change how reads and writes behave. That means attribute access is not merely “look in a dict and stop”.


In [1]:
class Demo:
    def __init__(self, x):
        self._x = x

    @property
    def x(self):
        return self._x

d = Demo(5)
print(d.__dict__)
print(Demo.__dict__["x"])
print(d.x)


{'_x': 5}
5


## What compiled bytecode reveals about this code

Disassembly of a property-using method reminds us that attribute access is a real interpreter operation. Python performs load-attribute steps that may trigger descriptor logic.


In [2]:
import dis

class Demo:
    def __init__(self, x):
        self._x = x
    @property
    def x(self):
        return self._x

def read_value(obj):
    return obj.x

dis.dis(read_value)


 10           0 RESUME                   0

 11           2 LOAD_FAST                0 (obj)
              4 LOAD_ATTR                0 (x)
             14 RETURN_VALUE


## A slower walk through the main ideas

### Attribute access has a search strategy
Instance data, class data, and descriptors all matter.

### Properties are a clean public interface
They let callers use attribute syntax while the class keeps control.

### Descriptors are the general mechanism
Properties, methods, and many framework features are built on this idea.

### Understanding the model makes advanced Python calmer
It turns surprising behavior into expected behavior.


## A property with validation

This keeps attribute-style use while still enforcing rules.


In [3]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = value

t = Temperature(25)
t.celsius = 30
print(t.celsius)


30


## A tiny descriptor

This custom object controls how the attribute behaves when read or written.


In [4]:
class PositiveNumber:
    def __set_name__(self, owner, name):
        self.private_name = "_" + name
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.private_name)
    def __set__(self, instance, value):
        if value <= 0:
            raise ValueError("must be positive")
        setattr(instance, self.private_name, value)

class Product:
    price = PositiveNumber()
    def __init__(self, price):
        self.price = price

p = Product(10)
print(p.price)


10


## Bound methods are descriptor behavior too

Methods becoming bound is part of the same broad story.


In [5]:
class Greeter:
    def hello(self):
        return "hi"

g = Greeter()
print(Greeter.hello)
print(g.hello)


<function Greeter.hello at 0x000001901D3144A0>
<bound method Greeter.hello of <__main__.Greeter object at 0x000001901D302110>>


## Where this usually shows up in real code

This is not only a classroom topic. **Attribute Lookup, Properties, and Descriptors** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


## Common traps that still catch careful people

- Thinking properties are only fancy getters and setters with no runtime model behind them
- Jumping into descriptors before understanding attribute lookup basics
- Using hidden heavy work behind properties without making it clear


## Questions that reveal whether this idea is really clear yet

- What is a property?
- What is a descriptor?
- Why do methods become bound on instances?


## What I would really want you to remember later

- Attribute access is a real lookup process.
- Properties are friendly descriptor-based tools.
- Descriptors explain a surprising amount of Python.


## One last grounding thought

If this notebook did its job, **Attribute Lookup, Properties, and Descriptors** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.
